In [ ]:
# Pseudocódigo Adaptado (Optimización Global de Puntos)

#     Entrada:
#         K: Tu nube de puntos (que ahora actúa como el conjunto de puntos finales a conectar).
#         sroot​: Punto de inicio definido manualmente (Arteria Hepática Propia).
#     Inicialización:
#         Crear la primera rama: Selecciona el punto de K más cercano a sroot​ y crea la rama raíz c0​=(sroot​,e0​).
#         C←{c0​} (Conjunto inicial de ramas).
#         Eliminar el punto utilizado de K.
#     Bucle de Conexión (Para cada punto restante pk​ en K):
#         Crear una cola de prioridad vacía q.
#         PARA CADA rama ci​ actualmente en C:
#             Calcular el punto de bifurcación óptimo (b).
#             Calcular el coste de volumen vascular total: coste=δ1​+δ2​+δ3​.
#             Añadir (i,coste) a la cola q.
#         Extraer la rama ci​ con el coste mínimo global (ya que no hay filtro de segmento).
#         Verificación:
#             SI las nuevas ramas resultantes de conectar pk​ a ci​ no se intersectan (chequeo GPU):
#                 Reemplazar ci​ en C por las tres nuevas ramas: (si​,b), (b,ei​) y (b,pk​).
#                 Pasar al siguiente punto de K.
#     Finalización (Geometría y Radios):
#         Asignar Radios: Aplicar la Ley de Murray (Rpadre2γ​=R12γ​+R22γ​) desde la raíz hacia los extremos.
#         Suavizado: Aplicar el ajuste polinómico cúbico a cada rama para generar curvas suaves y ángulos de bifurcación realistas basados en γ.

In [ ]:
def vessel_tree_optimizer(K, sroot, gamma=2/3):
    """
    Optimiza la conexión de puntos en K a partir de un punto raíz sroot.
    
    Args:
        K: Conjunto de puntos a conectar.
        sroot: Punto raíz (coordenadas del punto de entrada).
        gamma: Exponente de Murray (default 2/3).
    
    Returns:
        C: Conjunto de ramas del árbol vascular optimizado.
    """
    # Implementación del algoritmo aquí

    # Crear la primera rama
    closest_point = min(K, key=lambda p: np.linalg.norm(p - sroot))
    C = [(sroot, closest_point)]
    K.remove(closest_point)

    pass

In [6]:
import numpy as np

class Branch:
    def __init__(self, start, end, parent=None):
        self.start = np.array(start)
        self.end = np.array(end)
        self.parent = parent
        self.children = []
        self.Q = 1  # Número de ramas terminales (hojas) que alimenta [4]
        self.radius = 0.0

class HepaticVesselTree:
    def __init__(self, root_pos, gamma=3.0):
        # gamma (exponente de Murray) recomendado entre 2.8 y 3.1 [5, 6]
        self.gamma = gamma
        self.branches = []
        # Inicializamos con el primer segmento desde la raíz [7]
        self.root_pos = np.array(root_pos)

    def calculate_bifurcation_point(self, branch, p_k):
        """Implementa la Ecuación 1 del paper para el centroide b"""
        s_i = branch.start
        e_i = branch.end
        Q_i = branch.Q
        
        denom = 1 + (Q_i + 1)**self.gamma + Q_i**self.gamma
        # print("pk", p_k, "si", s_i, "ei", e_i, "Qi", Q_i, "denom", denom)
        b = (p_k + (Q_i + 1)**self.gamma * s_i + Q_i**self.gamma * e_i) / denom
        return b

    def calculate_cost(self, branch, p_k, b):
        """Calcula el coste de volumen vascular (Ecuaciones 2-5) [2]"""
        Q_i = branch.Q
        
        # Delta 1: Cambio en el segmento original al acortarse
        d1 = Q_i**self.gamma * (np.linalg.norm(branch.end - b)**2 - 
                                np.linalg.norm(branch.end - branch.start)**2)
        
        # Delta 2: Volumen de las dos nuevas ramas (hacia p_k y b)
        d2 = (Q_i + 1)**self.gamma * np.linalg.norm(b - branch.start)**2 + \
             np.linalg.norm(p_k - b)**2
        
        # Delta 3: Incremento de radio en todos los ancestros (padres)
        d3 = 0
        current = branch.parent
        while current is not None:
            Q_k = current.Q
            d3 += ((Q_k + 1)**self.gamma - Q_k**self.gamma) * \
                  np.linalg.norm(current.end - current.start)**2
            current = current.parent
            
        return d1 + d2 + d3

    def add_point(self, p_k):
        """Conecta un nuevo punto minimizando el coste global [8]"""
        if not self.branches:
            # Si es el primer punto, conectamos directamente a la raíz
            new_branch = Branch(self.root_pos, p_k)
            self.branches.append(new_branch)
            return

        best_cost = float('inf')
        best_branch = None
        best_b = None

        # Búsqueda global de la mejor rama (sin restricción de segmento) [8]
        for branch in self.branches:
            b = self.calculate_bifurcation_point(branch, p_k)
            cost = self.calculate_cost(branch, p_k, b)
            
            if cost < best_cost:
                best_cost = cost
                best_branch = branch
                best_b = b

        # Actualizar topología: Reemplazar rama original por 3 nuevas [1, 9]
        self._split_branch(best_branch, p_k, best_b)

    def _split_branch(self, branch, p_k, b):
        # 1. Rama desde el inicio original hasta la bifurcación
        # 2. Rama desde la bifurcación hasta el fin original
        # 3. Rama desde la bifurcación hasta el nuevo punto p_k
        orig_end = branch.end
        branch.end = b
        
        new_branch_orig = Branch(b, orig_end, parent=branch)
        new_branch_pk = Branch(b, p_k, parent=branch)
        
        self.branches.extend([new_branch_orig, new_branch_pk])
        
        # Actualizar Q (flujo) hacia arriba hasta la raíz [10]
        curr = branch
        while curr is not None:
            curr.Q += 1
            curr = curr.parent

    def compute_radii(self, R_root):
        """Calcula los radios basados en la Ley de Murray (Ecuación 6) [11]"""
        # Se asume que el árbol ya está construido. 
        # La relación es R_padre^(2*gamma) = R1^(2*gamma) + R2^(2*gamma)
        for b in reversed(self.branches):
            if not b.children:
                b.radius = R_root * (1.0 / self.branches.Q)**(1/(2*self.gamma))
            else:
                # Simplificación: el radio escala con Q^(1/gamma) [4]
                b.radius = R_root * (b.Q / self.branches.Q)**(1/(2*self.gamma))

In [7]:

# --- Ejemplo de uso ---
puntos_nube = [np.random.rand(3) * 100 for _ in range(10)] # Tu nube de puntos
root_manual = [12, 12, 12] # Punto de inicio (Hilum) [7]

tree = HepaticVesselTree(root_manual, gamma=3.0)

for p in puntos_nube:
    tree.add_point(p)

# tree.compute_radii(R_root=3.0) # Radio inicial de la arteria hepática [5] # TODO fix

print(f"Árbol generado con {len(tree.branches)} ramas.")

Árbol generado con 19 ramas.


In [17]:
# save tree on json format
import json
import numpy as np

def save_tree_to_json(tree, filename):
    tree_data = []
    for branch in tree.branches:
        branch_info = {
            'start': branch.start.tolist(),
            'end': branch.end.tolist(),
            'radius': branch.radius,
            'Q': branch.Q
        }
        tree_data.append(branch_info)

    with open(filename, 'w') as f:
        json.dump(tree_data, f, indent=4)

def _normalize_point_3d(point):
    arr = np.asarray(point).reshape(-1)
    if arr.size != 3:
        return None
    return [arr[0], arr[1], arr[2]]

def save_points_to_json(tree, filename):
    # Viewer expects: [[x, y, z], [x, y, z], ...]
    points_data = []
    seen = set()

    for branch in tree.branches:
        for point in (branch.start, branch.end):
            p3 = _normalize_point_3d(point)
            if p3 is None:
                continue
            key = tuple(p3)
            if key not in seen:
                seen.add(key)
                points_data.append(p3)

    with open(filename, 'w') as f:
        json.dump(points_data, f, indent=4)

save_tree_to_json(tree, 'vessel_tree.json')
save_points_to_json(tree, 'vessel_points.json')